### ViT‑U-Net (single cell)

This cell defines a Vision-Transformer U‑Net (hybrid) model, losses, and TF dataset helpers for 256×256×3 brain MRI segmentation. Paste/run this cell to build and compile the model; then create datasets and call `model.fit()` as needed.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers
import numpy as np
import os
from sklearn.model_selection import train_test_split

# ---------------------------
# Model building utilities
# ---------------------------

def conv_block(x, filters, kernel_size=3):
    x = layers.Conv2D(filters, kernel_size, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(filters, kernel_size, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    return x


class TransformerBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads=8, mlp_dim=512, dropout=0.0):
        super().__init__()
        self.attn = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.norm1 = layers.LayerNormalization(epsilon=1e-6)
        self.norm2 = layers.LayerNormalization(epsilon=1e-6)
        self.mlp = tf.keras.Sequential([
            layers.Dense(mlp_dim, activation=tf.nn.gelu),
            layers.Dropout(dropout),
            layers.Dense(embed_dim),
            layers.Dropout(dropout),
        ])

    def call(self, x, training=False):
        # x: (batch, seq_len, embed_dim)
        attn_out = self.attn(x, x)
        x = x + attn_out
        x = self.norm1(x)
        mlp_out = self.mlp(x, training=training)
        x = x + mlp_out
        x = self.norm2(x)
        return x


def build_vit_unet(
    input_shape=(256,256,3),
    num_classes=1,
    base_filters=32,
    depth=4,
    transformer_depth=4,
    num_heads=8,
    mlp_dim=512,
    patch_size=2,
):
    """Hybrid U-Net with Transformer bottleneck."""
    inputs = layers.Input(shape=input_shape)

    # Encoder (save skip connections)
    skips = []
    x = inputs
    for i in range(depth):
        filters = base_filters * (2**i)
        x = conv_block(x, filters)
        skips.append(x)
        x = layers.MaxPool2D(2)(x)

    # Bottleneck conv
    filters = base_filters * (2**depth)
    x = conv_block(x, filters)

    # Patch embedding (Conv with stride=patch_size)
    # We assume the bottleneck spatial dims are divisible by patch_size (for 256 input and our design they will be)
    p = layers.Conv2D(filters, kernel_size=patch_size, strides=patch_size, padding='valid')(x)
    h_p, w_p = p.shape[1], p.shape[2]
    seq_len = (h_p or 1) * (w_p or 1)
    embed_dim = p.shape[-1]

    # Flatten to sequence
    flat = layers.Reshape((seq_len, embed_dim))(p)

    # Positional embeddings (learnable)
    # create a trainable positional embedding tensor of shape (1, seq_len, embed_dim)
    pos_embed = tf.Variable(initial_value=tf.random.truncated_normal([1, seq_len, embed_dim], stddev=0.02), trainable=True, name='pos_embed')
    flat = flat + pos_embed

    # Transformer blocks
    for _ in range(transformer_depth):
        flat = TransformerBlock(embed_dim, num_heads=num_heads, mlp_dim=mlp_dim)(flat)

    # Project back to spatial map
    x = layers.Reshape((h_p, w_p, embed_dim))(flat)
    if patch_size > 1:
        x = layers.UpSampling2D(size=patch_size, interpolation='nearest')(x)
        # match channels
        if x.shape[-1] != filters:
            x = layers.Conv2D(filters, 1, padding='same')(x)

    # Decoder
    for i in reversed(range(depth)):
        skip = skips[i]
        filters = base_filters * (2**i)
        x = layers.UpSampling2D(size=2, interpolation='nearest')(x)
        # if spatial sizes mismatch due to rounding, center-crop/pad could be added. For typical sizes this matches.
        x = layers.Concatenate()([x, skip])
        x = conv_block(x, filters)

    # Output
    activation = 'sigmoid' if num_classes == 1 else 'softmax'
    outputs = layers.Conv2D(num_classes, 1, padding='same', activation=activation)(x)

    model = tf.keras.Model(inputs, outputs, name='vit_unet')
    return model

# ---------------------------
# Losses / Metrics
# ---------------------------

def dice_coef(y_true, y_pred, smooth=1e-6):
    y_true_f = tf.reshape(y_true, [-1])
    y_pred_f = tf.reshape(y_pred, [-1])
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    return (2.0 * intersection + smooth) / (tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) + smooth)

def bce_dice_loss(y_true, y_pred):
    bce = tf.keras.losses.BinaryCrossentropy()(y_true, y_pred)
    return bce + (1.0 - dice_coef(y_true, y_pred))

# ---------------------------
# Data helpers (tf.data)
# ---------------------------

def load_image_mask_pair(image_path, mask_path, img_size=(256,256)):
    image = tf.io.read_file(image_path)
    image = tf.image.decode_image(image, channels=3)
    image = tf.image.resize(image, img_size)
    image = tf.cast(image, tf.float32) / 255.0

    mask = tf.io.read_file(mask_path)
    mask = tf.image.decode_image(mask, channels=1)
    mask = tf.image.resize(mask, img_size, method=tf.image.ResizeMethod.NEAREST_NEIGHBOR)
    # convert to binary mask (0/1)
    mask = tf.cast(mask > 127, tf.float32)
    return image, mask


def make_dataset(image_paths, mask_paths, batch_size=8, shuffle=True):
    ds = tf.data.Dataset.from_tensor_slices((image_paths, mask_paths))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(image_paths))

    def _map(i, m):
        return load_image_mask_pair(i, m)

    ds = ds.map(_map, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

# ---------------------------
# Example usage (build + compile)
# ---------------------------
if __name__ == '__main__':
    # Build & compile model
    model = build_vit_unet(input_shape=(256,256,3), num_classes=1, base_filters=32,
                          depth=4, transformer_depth=4, num_heads=8, mlp_dim=512, patch_size=2)
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-4), loss=bce_dice_loss, metrics=[dice_coef])
    model.summary()

    # Example: prepare dataset paths (adjust directories)
    # images_dir = r'C:\path\to\images'
    # masks_dir  = r'C:\path\to\masks'
    # image_files = sorted([os.path.join(images_dir, f) for f in os.listdir(images_dir)])
    # mask_files  = sorted([os.path.join(masks_dir, f) for f in os.listdir(masks_dir)])
    # (You may need a custom matching function depending on your filenames.)
    # train_i, val_i, train_m, val_m = train_test_split(image_files, mask_files, test_size=0.15, random_state=42)
    # train_ds = make_dataset(train_i, train_m, batch_size=8, shuffle=True)
    # val_ds = make_dataset(val_i, val_m, batch_size=8, shuffle=False)
    # model.fit(train_ds, validation_data=val_ds, epochs=30)
